# Agent Handoffs - Multi-Agent Routing

## Overview
Learn how to create specialized agents and route between them using handoffs for intelligent task delegation.

## What You'll Learn
- Creating specialist agents with specific expertise
- Using handoff_description to guide routing
- Building triage agents that delegate work
- Understanding automatic routing logic

## Key Concepts
- **Handoffs**: Passing control from one agent to another based on expertise
- **Triage Agent**: Routes incoming requests to the appropriate specialist
- **handoff_description**: Tells the triage agent when to hand off to this agent
- **Specialist Agents**: Focused agents with domain-specific instructions

## Step 1: Install Dependencies

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

In [ ]:
model_id = "openai.gpt-5.5"

## Step 2: Configure Authentication

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Step 3: Import Required Classes

In [ ]:
import asyncio

from agents import Agent, Runner

## Step 4: Create Specialist Agents

Define agents with specific expertise. Each specialist has:
- **name**: Identifies the agent
- **handoff_description**: Tells the triage agent when to use this specialist
- **instructions**: Domain-specific guidance

💡 **Key Point**: The `handoff_description` is read by the triage agent to decide routing!

🔍 **Watch**: Notice how each specialist has focused instructions for their domain.

In [ ]:
history_tutor = Agent(
    name="History tutor",
    model=model_id,
    handoff_description="Specialist for history questions.",  # Routing hint
    instructions="Answer history questions clearly and concisely.",
)

math_tutor = Agent(
    name="Math tutor",
    model=model_id,
    handoff_description="Specialist for math questions.",  # Routing hint
    instructions="Explain math step by step and include worked examples.",
)

## Step 5: Create Triage Agent

The triage agent coordinates specialists using the `handoffs` parameter.

**How it works:**
1. User sends a question to the triage agent
2. Triage agent reads all `handoff_description` fields
3. Decides which specialist is best suited
4. Automatically hands off to that specialist
5. Specialist processes and returns the answer

⚡ **Important**: The triage agent's instructions should mention routing to specialists!

In [ ]:
triage_agent = Agent(
    name="Homework triage",
    model=model_id,
    instructions="Route each homework question to the right specialist.",
    handoffs=[history_tutor, math_tutor],  # Available specialists
)

## Step 6: Run with Automatic Routing

Send a question to the triage agent and watch it automatically route to the correct specialist.

🎯 **Result**: This history question will be routed to the `history_tutor` automatically!

In [ ]:
result = await Runner.run(triage_agent, "Who was the first president of the United States?")
print(result.final_output)

## 🎓 Key Takeaways

- **Handoffs** enable automatic routing between specialized agents
- **handoff_description** is the key to routing logic - write it clearly!
- **Triage agents** coordinate work without needing domain expertise
- **Specialists** can focus on their specific domain with tailored instructions
- **Scalable**: Easy to add new specialists without changing triage logic
- **Automatic**: The SDK handles the routing - you just define the specialists